# Getting Started with LlamaCloud Indexing

This notebook is a comprehensive tutorial on using the Index feature in LlamaCloud to build powerful agentic applications with retrieval. 

In this example we use Apple and Tesla 10-K filings from 2019-2023. We show you how to use the `LlamaCloudIndex` class from `llama_cloud_services` for retrieval.

## Structure

**🚀 Getting Started with the Basics** 
Learn the fundamentals of LlamaCloudIndex:
1. **Index Creation**: Setting up a LlamaCloudIndex with 10 documents (5 Apple + 5 Tesla 10-Ks)
2. **Chunk Retrieval**: Standard retrieval for specific information
3. **Simple Query Engine**: Building basic RAG without complexity
4. **Basic Agent**: Simple agent with conversational capability

**🎯 Advanced Features**
Explore more sophisticated patterns:
1. **File-Level Retrieval**: Retrieving entire documents for comprehensive analysis
2. **Smart Agent**: Agent that can choose between chunk and file retrieval
3. **Query Engine with Citations**: Enhanced RAG with source citations

**Status:**
| Last Updated | Version | State      |
|--------------|---------|------------|
| Jan-2025     | 0.1.0   | Active     |


## Setup and Installation

First, let's install the required packages and set up our environment.


In [ ]:
# Install required packages
%pip install "llama-index>=0.13.0<0.14.0" llama-cloud-services
%pip install llama-index-llms-openai llama-index-embeddings-openai


In [ ]:
# Setup for async execution in notebooks
import nest_asyncio
nest_asyncio.apply()

import os
from IPython.display import Markdown, display


## API Keys Setup

Set your API keys. You'll need:
- **LlamaCloud API Key**: Get from [cloud.llamaindex.ai](https://cloud.llamaindex.ai)
- **OpenAI API Key**: For LLM and embedding models


In [ ]:
# Set your API keys
os.environ["LLAMA_CLOUD_API_KEY"] = "llx-..."
os.environ["OPENAI_API_KEY"] = "sk-..."


## Data Download

Let's download the Apple and Tesla 10-K filings for 2019-2023. These are publicly available SEC filings that provide comprehensive annual business information.


In [ ]:
# Create data directory
!mkdir -p data


In [ ]:
# Download Apple 10-K filings (2019-2023)
print("Downloading Apple 10-K filings...")
!wget "https://s2.q4cdn.com/470004039/files/doc_earnings/2023/q4/filing/_10-K-Q4-2023-As-Filed.pdf" -O data/apple_2023.pdf
!wget "https://s2.q4cdn.com/470004039/files/doc_financials/2022/q4/_10-K-2022-(As-Filed).pdf" -O data/apple_2022.pdf
!wget "https://s2.q4cdn.com/470004039/files/doc_financials/2021/q4/_10-K-2021-(As-Filed).pdf" -O data/apple_2021.pdf
!wget "https://s2.q4cdn.com/470004039/files/doc_financials/2020/ar/_10-K-2020-(As-Filed).pdf" -O data/apple_2020.pdf
!wget "https://www.dropbox.com/scl/fi/i6vk884ggtq382mu3whfz/apple_2019_10k.pdf?rlkey=eudxh3muxh7kop43ov4bgaj5i&dl=1" -O data/apple_2019.pdf

print("Downloading Tesla 10-K filings...")
# Download Tesla 10-K filings (2019-2023)
!wget "https://ir.tesla.com/_flysystem/s3/sec/000162828024002390/tsla-20231231-gen.pdf" -O data/tesla_2023.pdf
!wget "https://ir.tesla.com/_flysystem/s3/sec/000095017023001409/tsla-20221231-gen.pdf" -O data/tesla_2022.pdf
!wget "https://www.dropbox.com/scl/fi/ptk83fmye7lqr7pz9r6dm/tesla_2021_10k.pdf?rlkey=24kxixeajbw9nru1sd6tg3bye&dl=1" -O data/tesla_2021.pdf
!wget "https://ir.tesla.com/_flysystem/s3/sec/000156459021004599/tsla-10k_20201231-gen.pdf" -O data/tesla_2020.pdf
!wget "https://ir.tesla.com/_flysystem/s3/sec/000156459020004475/tsla-10k_20191231-gen_0.pdf" -O data/tesla_2019.pdf

print("\nDownload complete! Files available:")
!ls -la data/


## 1. Creating a LlamaCloudIndex

Now we'll create a LlamaCloudIndex by uploading our documents. LlamaCloudIndex provides a managed indexing solution that handles:
- Document parsing and chunking
- Embedding generation
- Hybrid search (dense + sparse)
- Advanced retrieval with reranking


In [ ]:
from llama_cloud_services import LlamaCloudIndex
from llama_index.core import Document
import glob

# Load all PDF documents
pdf_files = glob.glob("data/*.pdf")
documents = []

for pdf_file in sorted(pdf_files):
    # Create Document objects with metadata
    company = "Apple" if "apple" in pdf_file.lower() else "Tesla"
    year = pdf_file.split("_")[-1].replace(".pdf", "")
    
    doc = Document(
        file_path=pdf_file,
        metadata={
            "company": company,
            "year": year,
            "document_type": "10-K",
            "file_name": os.path.basename(pdf_file)
        }
    )
    documents.append(doc)

print(f"Loaded {len(documents)} documents:")
for doc in documents:
    print(f"  - {doc.metadata['company']} {doc.metadata['year']} 10-K")


In [ ]:
# Create the LlamaCloudIndex
print("Creating LlamaCloudIndex... This may take a few minutes.")

index = LlamaCloudIndex.from_documents(
    documents,
    name="apple_tesla_10k_demo",
    project_name="llamacloud_demo",
    show_progress=True
)

print("\n✅ Index created successfully!")
print(f"Index ID: {index.pipeline_id}")


## 🚀 Getting Started with the Basics

Now that we have our index, let's explore the fundamental patterns for building RAG applications.

### Chunk-Level Retrieval

Let's start with the most common retrieval pattern - finding relevant chunks of text from our documents.


The chunk-level retriever performs hybrid search (dense + sparse) with reranking to find the most relevant pieces of information.


In [ ]:
# Create a chunk-level retriever
chunk_retriever = index.as_retriever(
    dense_similarity_top_k=5,
    sparse_similarity_top_k=5,
    enable_reranking=True,
    rerank_top_n=3
)

# Example query
query = "What are the main revenue sources for Apple and Tesla?"
chunk_nodes = chunk_retriever.retrieve(query)

print(f"Retrieved {len(chunk_nodes)} chunks for query: '{query}'\n")

for i, node in enumerate(chunk_nodes):
    print(f"**Chunk {i+1}** (Score: {node.score:.3f})")
    print(f"Source: {node.metadata.get('file_name', 'Unknown')}")
    print(f"Content: {node.text[:300]}...\n")


### Simple Query Engine

Now let's build a basic RAG query engine that combines retrieval with response generation.


In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

# Set up LLM
llm = OpenAI(model="gpt-5-mini", temperature=0.1)
Settings.llm = llm

# Create a simple query engine
query_engine = RetrieverQueryEngine.from_args(
    chunk_retriever,
    llm=llm,
    response_mode="compact"
)

print("🤖 **Simple RAG Query Engine**\n")


Let's test our query engine with some basic questions about Apple and Tesla:


In [ ]:
# Test with simple questions
basic_queries = [
    "What was Apple's total revenue in 2023?",
    "What are Tesla's main products?",
    "How much did Apple spend on R&D in 2022?"
]

for query in basic_queries:
    print(f"**Q: {query}**")
    response = query_engine.query(query)
    print(f"**A:** {response}\n")
    print("-" * 60 + "\n")


### Basic Agent

Let's create a simple agent that can have conversations and maintain memory while answering questions about our documents.


In [ ]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.memory import ChatMemoryBuffer

# Create a tool for the agent - single tool for both companies
document_tool = QueryEngineTool(
    query_engine=query_engine,
    metadata=ToolMetadata(
        name="document_search",
        description="Search and analyze Apple and Tesla 10-K filings (2019-2023). Use for questions about either company's business, financials, products, or strategy."
    )
)

# Create the agent using FunctionAgent with memory
memory = ChatMemoryBuffer.from_defaults(token_limit=3000)
agent = FunctionAgent(
    tools=[document_tool],
    llm=llm,
    system_prompt="You are a helpful assistant that can search and analyze financial documents from Apple and Tesla. Use the available tools to answer questions accurately."
)

print("🤖 **Basic Agent Ready!**")
print("The agent can answer questions about both Apple and Tesla using FunctionAgent.")


Let's test the agent with a conversational flow where it remembers previous questions:


In [ ]:
# Example conversation showing the agent capabilities
conversation = [
    "What was Apple's revenue in 2023?",
    "How does that compare to Tesla's revenue?",
    "Which company grew faster between 2022 and 2023?",
    "What do you think is the main reason for the difference?"
]

print("💬 **Agent Conversation**\n")

import asyncio

async def run_conversation():
    for i, query in enumerate(conversation, 1):
        print(f"**Turn {i}: {query}**")
        response = await agent.run(query, memory=memory)
        print(f"**Agent:** {response}\n")
        print("-" * 60 + "\n")

# Run the conversation
await run_conversation()


## 🎯 Advanced Features

Now let's explore more sophisticated patterns that showcase the full power of LlamaCloudIndex.

### File-Level Retrieval

For questions requiring comprehensive document context, file-level retrieval returns entire documents rather than just chunks.


In [ ]:
# Create a file-level retriever
file_retriever = index.as_retriever(
    retrieval_mode="files_via_content",  # or "files_via_metadata"
    files_top_k=2
)

# Example query that benefits from full document context
query = "Give me a comprehensive analysis of Tesla's business model evolution from 2019 to 2023"
file_nodes = file_retriever.retrieve(query)

print(f"Retrieved {len(file_nodes)} documents for query: '{query}'\n")

for i, node in enumerate(file_nodes):
    print(f"**Document {i+1}**")
    print(f"Source: {node.metadata.get('file_name', 'Unknown')}")
    print(f"Content length: {len(node.text):,} characters")
    print(f"First 300 chars: {node.text[:300]}...\n")


### Smart Agent with Multiple Retrieval Strategies

Let's create an advanced agent that can choose between chunk-level and file-level retrieval based on the question type.


This agent will have two tools - one for specific factual queries (chunk-level) and one for comprehensive analysis (file-level).


In [ ]:
# Create query engines for both retrieval strategies
chunk_query_engine = RetrieverQueryEngine.from_args(
    chunk_retriever,
    llm=llm,
    response_mode="compact"
)

file_query_engine = RetrieverQueryEngine.from_args(
    file_retriever, 
    llm=llm,
    response_mode="tree_summarize"  # Better for large documents
)

# Create tools for the smart agent
smart_tools = [
    QueryEngineTool(
        query_engine=chunk_query_engine,
        metadata=ToolMetadata(
            name="specific_search",
            description="Search for specific facts, numbers, or targeted information from Apple and Tesla 10-K filings. Best for precise factual queries."
        )
    ),
    QueryEngineTool(
        query_engine=file_query_engine,
        metadata=ToolMetadata(
            name="comprehensive_analysis",
            description="Perform comprehensive document-level analysis requiring full context. Use for complex questions, comparisons, or broad business analysis."
        )
    )
]

# Create the smart agent using FunctionAgent
smart_agent = FunctionAgent(
    tools=smart_tools,
    llm=llm,
    system_prompt="You are an intelligent assistant with access to two search strategies. Use 'specific_search' for targeted factual queries and 'comprehensive_analysis' for complex analysis requiring full document context. Choose the appropriate tool based on the question type."
)

print("🧠 **Smart Agent Ready!**")
print("The agent can choose between:")
print("  - Specific search (chunk-level) for targeted queries")
print("  - Comprehensive analysis (file-level) for complex questions")


Let's test queries that should trigger different retrieval strategies:


In [ ]:
# Test different types of queries to see tool selection
test_queries = [
    "What was Apple's exact revenue in 2023?",  # Should use specific_search
    "Analyze Tesla's overall business transformation from 2019 to 2023",  # Should use comprehensive_analysis
    "Compare the strategic positioning of Apple vs Tesla over the past 5 years"  # Should use comprehensive_analysis
]

print("🧠 **Smart Agent Tool Selection Test**\n")

async def run_smart_tests():
    smart_memory = ChatMemoryBuffer.from_defaults(token_limit=3000)
    for i, query in enumerate(test_queries, 1):
        print(f"**Query {i}: {query}**")
        response = await smart_agent.run(query, memory=smart_memory)
        print(f"**Agent:** {response}\n")
        print("=" * 80 + "\n")

# Run the smart agent tests
await run_smart_tests()


### Query Engine with Citations

For transparency and verification, let's create a query engine that provides inline citations linking back to source documents.


In [ ]:
# Define citation components
from typing import List, Optional
from llama_index.core import QueryBundle
from llama_index.core.postprocessor.types import BaseNodePostprocessor
from llama_index.core.schema import NodeWithScore

class NodeCitationProcessor(BaseNodePostprocessor):
    """Add node_id to metadata for citation linking."""
    
    def _postprocess_nodes(
        self,
        nodes: List[NodeWithScore],
        query_bundle: Optional[QueryBundle] = None,
    ) -> List[NodeWithScore]:
        for node_score in nodes:
            node_score.node.metadata["node_id"] = node_score.node.node_id
        return nodes

# Citation system prompt
SYSTEM_CITATION_PROMPT = """You have provided information from a knowledge base that has been passed to you in nodes of information.
Each node has useful metadata such as node ID, file name, page, etc.
Please add the citation to the data node for each sentence or paragraph that you reference in the provided information.
The citation format is: [citation:<node_id>]()
Where the <node_id> is the unique identifier of the data node.

Example:
We have two nodes:
  node_id: xyz
  file_name: apple_2023.pdf
  
  node_id: abc
  file_name: tesla_2022.pdf

User question: Tell me about Apple's revenue.
Your answer:
Apple's total revenue was $383.3 billion in fiscal 2023 [citation:xyz]().
This represents strong growth compared to previous years [citation:xyz]()."""


In [ ]:
# Create query engine with citations
citation_llm = OpenAI(model="gpt-4o-mini", system_prompt=SYSTEM_CITATION_PROMPT)

citation_query_engine = RetrieverQueryEngine.from_args(
    chunk_retriever,
    llm=citation_llm,
    response_mode="tree_summarize",
    node_postprocessors=[NodeCitationProcessor()]
)

# Function to process citations and create readable links
import re

def process_citations_with_sources(response) -> str:
    content = str(response)
    source_nodes = response.source_nodes

    # Create a lookup: citation_id -> file info
    id_to_file = {
        str(node.id_): node.metadata.get('file_name', 'unknown')
        for node in source_nodes
    }

    # Track citation order and assign human-friendly numbers
    citation_order = {}
    citation_counter = 1

    def replace(match):
        nonlocal citation_counter
        citation_id = match.group(1).strip()
        if citation_id not in citation_order:
            citation_order[citation_id] = citation_counter
            citation_counter += 1
        number = citation_order[citation_id]
        file_name = id_to_file.get(citation_id, 'unknown')
        return f"[{number}]({file_name})"

    # Replace citations with numbered references
    citation_regex = re.compile(r'\[citation:([^\]]+)\]')
    content = citation_regex.sub(replace, content)

    # Remove incomplete citation tags
    incomplete_regex = re.compile(r'\[citation:[^\]]*$')
    content = incomplete_regex.sub('', content)

    return content

print("📚 **Citations Query Engine Ready!**")


In [ ]:
# Test queries with citations
citation_queries = [
    "What were Apple's main revenue sources in 2023?",
    "How did Tesla's automotive sales perform in 2022?"
]

print("📚 **Query Engine with Citations**\n")

for query in citation_queries:
    print(f"**Q: {query}**")
    response = citation_query_engine.query(query)
    
    # Process and display the response with citations
    content_with_citations = process_citations_with_sources(response)
    print(f"**A:** {content_with_citations}\n")
    print("-" * 60 + "\n")


## Summary

In this notebook, we've explored LlamaCloudIndex capabilities in a structured way:

### 🚀 Basics Covered:
1. **✅ Index Creation**: Built a managed index with 10 financial documents
2. **✅ Chunk Retrieval**: Standard retrieval for specific information
3. **✅ Simple Query Engine**: Basic RAG without complexity
4. **✅ Basic Agent**: FunctionAgent with conversational capability

### 🎯 Advanced Features:
1. **✅ File-Level Retrieval**: Comprehensive document-level analysis
2. **✅ Smart Agent**: Intelligent tool selection between retrieval strategies
3. **✅ Citations**: Enhanced transparency with source linking

### Key Takeaways:

- **Progressive Learning**: Start simple, then add complexity
- **LlamaCloudIndex** provides enterprise-grade document search
- **Multiple Retrieval Modes** optimize for different query types
- **FunctionAgent Integration** uses LLM's native function calling for efficiency
- **Citations** add transparency and trust

### Next Steps:

- Experiment with different retrieval parameters
- Try other document types or domains
- Build custom workflows and integrations
- Explore advanced agent patterns

For more examples and documentation:
- [LlamaCloudIndex Documentation](https://docs.cloud.llamaindex.ai)
- [LlamaIndex Core Documentation](https://docs.llamaindex.ai)
